# 🧠 Autoencoders & Variational Autoencoders (VAE)
### Dimensionality Reduction & Deep Generative Models

---

**What you'll learn in this notebook:**
- Build a classic Autoencoder from scratch
- Visualize compressed (latent) representations
- Build a Variational Autoencoder (VAE)
- Understand the KL divergence loss
- Generate new images using the VAE
- Compare AE vs VAE outputs side by side

**Dataset**: MNIST (handwritten digits — 70,000 images, 28×28 pixels)

> 💡 **Framework**: We use **TensorFlow/Keras** here. PyTorch equivalents are noted in comments where key differences exist.

---

## 📦 Step 0: Install & Import Dependencies

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.losses import binary_crossentropy

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("All imports successful! ✅")

---
## 📂 Step 1: Load & Preprocess MNIST Data

**Why MNIST?**  
MNIST is the 'Hello World' of image datasets — 70,000 grayscale images of handwritten digits (0–9), each 28×28 pixels. It's perfect for learning because:
- Small enough to train quickly on CPU
- Complex enough to show meaningful compression
- Easy to visualize results

**Preprocessing steps:**
1. **Normalize** pixel values from [0, 255] → [0, 1] (helps the network learn faster)
2. **Flatten** images from 28×28 → 784-dimensional vectors (for our Dense-layer models)
3. We use the **same data for input and output** — autoencoders learn to reconstruct themselves!

In [ ]:
# Load MNIST dataset (downloads automatically on first run)
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize pixel values to [0, 1]
# Original values are integers in [0, 255]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32') / 255.0

# Flatten 28x28 images into 784-dim vectors
# Shape: (60000, 28, 28) → (60000, 784)
x_train_flat = x_train.reshape(-1, 784)
x_test_flat  = x_test.reshape(-1, 784)

print(f"Training samples : {x_train_flat.shape}")
print(f"Test samples     : {x_test_flat.shape}")
print(f"Pixel range      : [{x_train_flat.min():.1f}, {x_train_flat.max():.1f}]")

In [ ]:
# Visualize a few samples from the dataset
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i in range(10):
    axes[i].imshow(x_train[i], cmap='gray')
    axes[i].set_title(f'Label: {y_train[i]}', fontsize=8)
    axes[i].axis('off')
plt.suptitle('Sample MNIST Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🏗️ Step 2: Build a Classic Autoencoder

### Architecture
```
Input (784)  →  Encoder  →  Latent (2D)  →  Decoder  →  Output (784)
```

We use **2D latent space** intentionally — so we can visualize it on a 2D scatter plot later!

**Encoder**: 784 → 256 → 64 → **2** (bottleneck)  
**Decoder**: 2 → 64 → 256 → **784** (reconstruction)

> 🔑 **Key idea**: The bottleneck (2D) forces the network to learn the most important features of the data. It can't just memorize — it has to compress smartly.

In [ ]:
# ─────────────────────────────────────────
#  AUTOENCODER ARCHITECTURE
# ─────────────────────────────────────────

LATENT_DIM = 2  # 2D so we can visualize the latent space

# ── ENCODER ──
ae_input = keras.Input(shape=(784,), name='ae_input')
x = layers.Dense(256, activation='relu', name='enc_dense1')(ae_input)
x = layers.Dense(64,  activation='relu', name='enc_dense2')(x)
ae_latent = layers.Dense(LATENT_DIM, activation='linear', name='latent')(x)
# Note: linear activation at bottleneck lets values be any real number

# ── DECODER ──
x = layers.Dense(64,  activation='relu', name='dec_dense1')(ae_latent)
x = layers.Dense(256, activation='relu', name='dec_dense2')(x)
ae_output = layers.Dense(784, activation='sigmoid', name='ae_output')(x)
# Sigmoid output: keeps pixel values in [0, 1]

# ── FULL MODEL ──
autoencoder = Model(ae_input, ae_output, name='Autoencoder')
autoencoder.compile(
    optimizer='adam',
    loss='binary_crossentropy'  # Good for pixel-level reconstruction
    # PyTorch equivalent: nn.BCELoss()
)

autoencoder.summary()

---
## 🏋️ Step 3: Train the Autoencoder

Notice: **input = output = x_train_flat**  
The model learns to reconstruct its own input — that's what makes it an autoencoder!

**Hyperparameters:**
- `epochs=20`: 20 full passes through training data
- `batch_size=256`: process 256 images at a time
- `validation_split=0.1`: keep 10% of data to monitor overfitting

In [ ]:
# Train the Autoencoder
# Input = Output = x_train_flat (self-supervised!)
ae_history = autoencoder.fit(
    x_train_flat, x_train_flat,   # input and target are the SAME
    epochs=20,
    batch_size=256,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

In [ ]:
# Plot training loss curves
plt.figure(figsize=(8, 4))
plt.plot(ae_history.history['loss'],     label='Train Loss', color='steelblue', linewidth=2)
plt.plot(ae_history.history['val_loss'], label='Val Loss',   color='coral',     linewidth=2, linestyle='--')
plt.title('Autoencoder — Training Loss', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 🖼️ Step 4: Visualize AE Reconstruction

We'll compare **original** vs **reconstructed** images.  
If the autoencoder learned well, reconstructions should look very similar to originals — maybe slightly blurry, but recognizable.

In [ ]:
# Generate reconstructions from test set
ae_reconstructed = autoencoder.predict(x_test_flat, verbose=0)

# Helper function to display original vs reconstruction
def plot_reconstruction(originals, reconstructed, model_name, n=10):
    fig, axes = plt.subplots(2, n, figsize=(16, 3.5))
    fig.suptitle(f'{model_name} — Original vs Reconstructed', fontsize=13, fontweight='bold')
    
    for i in range(n):
        # Top row: original images
        axes[0, i].imshow(originals[i].reshape(28, 28), cmap='gray')
        axes[0, i].axis('off')
        if i == 0: axes[0, i].set_title('Original', fontsize=9, loc='left')
        
        # Bottom row: reconstructed images
        axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
        axes[1, i].axis('off')
        if i == 0: axes[1, i].set_title('Reconstructed', fontsize=9, loc='left')
    
    plt.tight_layout()
    plt.show()

plot_reconstruction(x_test_flat, ae_reconstructed, 'Autoencoder')

---
## 🗺️ Step 5: Visualize the AE Latent Space

Since we used a **2D latent space**, we can plot every test image as a dot on a 2D chart, colored by its digit label.

**What to look for:**  
- Well-trained AE: digits cluster together (all 7s in one region, all 0s in another)
- But: there may be **gaps** between clusters — those gaps contain *no valid digits*, which is a limitation of plain AEs

In [ ]:
# Build a standalone encoder model (just the encoder part)
ae_encoder_model = Model(ae_input, ae_latent, name='AE_Encoder')

# Encode all test images into 2D latent space
z_ae = ae_encoder_model.predict(x_test_flat, verbose=0)

# Scatter plot: each point = one test image, color = digit label
plt.figure(figsize=(8, 6))
scatter = plt.scatter(z_ae[:, 0], z_ae[:, 1], 
                      c=y_test, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit Label')
plt.title('Autoencoder — 2D Latent Space', fontsize=13, fontweight='bold')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 Notice: clusters may not be smoothly connected — gaps exist between digit groups.")
print("   If you tried to generate new digits by sampling a gap, you'd get garbage.")

---
## 🎲 Step 6: Build the Variational Autoencoder (VAE)

### What's Different from a Regular AE?

| Component | Autoencoder | VAE |
|-----------|-------------|-----|
| Encoder output | Single point z | Two vectors: **μ** (mean) and **log σ²** (log variance) |
| Sampling | Deterministic | **Stochastic** (uses reparameterization trick) |
| Loss | Reconstruction only | Reconstruction + **KL Divergence** |

### The Reparameterization Trick
We can't backpropagate through a random sampling step. So instead of sampling z directly:
```
z ~ N(μ, σ²)   ← NOT differentiable
```
We rephrase it as:
```
ε ~ N(0, 1)    ← random noise (no gradients needed here)
z = μ + σ * ε  ← differentiable! gradients flow through μ and σ
```

This is the key mathematical insight that makes VAE training possible.

In [ ]:
# ─────────────────────────────────────────
#  SAMPLING LAYER (Reparameterization Trick)
# ─────────────────────────────────────────

class Sampling(layers.Layer):
    """
    Custom Keras layer implementing the reparameterization trick.
    
    Input: [z_mean, z_log_var]  — both shape (batch, latent_dim)
    Output: z  — sampled latent vector, shape (batch, latent_dim)
    
    Formula: z = z_mean + exp(0.5 * z_log_var) * epsilon
             where epsilon ~ N(0, I)
    
    PyTorch equivalent:
        def reparameterize(self, mu, log_var):
            std = torch.exp(0.5 * log_var)
            eps = torch.randn_like(std)
            return mu + eps * std
    """
    def call(self, inputs):
        z_mean, z_log_var = inputs
        
        # Get the batch size and latent dim dynamically
        batch = tf.shape(z_mean)[0]
        dim   = tf.shape(z_mean)[1]
        
        # Sample standard normal noise: ε ~ N(0, I)
        epsilon = tf.random.normal(shape=(batch, dim))
        
        # Reparameterization: z = μ + σ * ε
        # Note: z_log_var = log(σ²), so σ = exp(0.5 * log(σ²))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

print("Sampling layer defined ✅")

In [ ]:
# ─────────────────────────────────────────
#  VAE ENCODER
# ─────────────────────────────────────────

VAE_LATENT_DIM = 2  # Keep 2D for visualization

# Input layer
vae_input = keras.Input(shape=(784,), name='vae_input')

# Shared encoder backbone
h = layers.Dense(256, activation='relu', name='vae_enc1')(vae_input)
h = layers.Dense(64,  activation='relu', name='vae_enc2')(h)

# Two output heads: mean and log-variance
z_mean    = layers.Dense(VAE_LATENT_DIM, name='z_mean')(h)
z_log_var = layers.Dense(VAE_LATENT_DIM, name='z_log_var')(h)

# Sampling layer applies reparameterization trick
z = Sampling(name='z_sampling')([z_mean, z_log_var])

# Wrap as a Keras Model (useful for encoding only during inference)
vae_encoder = Model(vae_input, [z_mean, z_log_var, z], name='VAE_Encoder')
vae_encoder.summary()

In [ ]:
# ─────────────────────────────────────────
#  VAE DECODER
# ─────────────────────────────────────────

# Decoder takes a latent vector z and reconstructs the image
latent_input = keras.Input(shape=(VAE_LATENT_DIM,), name='latent_input')

d = layers.Dense(64,  activation='relu', name='vae_dec1')(latent_input)
d = layers.Dense(256, activation='relu', name='vae_dec2')(d)
vae_output = layers.Dense(784, activation='sigmoid', name='vae_output')(d)

vae_decoder = Model(latent_input, vae_output, name='VAE_Decoder')
vae_decoder.summary()

---
## 🧮 Step 7: Define VAE with KL Divergence Loss

The VAE loss has **two parts**:

1. **Reconstruction Loss**: How well does the decoded output match the input?  
   `loss_recon = binary_crossentropy(original, reconstructed) × 784`

2. **KL Divergence Loss**: How close is the learned distribution N(μ, σ²) to a standard normal N(0, 1)?  
   `loss_kl = -0.5 × Σ(1 + log σ² - μ² - σ²)`

**Total loss = Reconstruction Loss + KL Divergence Loss**

> 🔑 **Why KL divergence?** Without it, the encoder could push encodings anywhere in the space. KL forces the latent space to stay *organized* around N(0,1), making generation reliable.

In [ ]:
# ─────────────────────────────────────────
#  FULL VAE MODEL WITH CUSTOM LOSS
# ─────────────────────────────────────────

class VAE(Model):
    """
    Full VAE model that combines encoder and decoder,
    and adds the KL divergence to the loss during training.
    """
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        
        # Keras metric trackers
        self.total_loss_tracker   = keras.metrics.Mean(name='total_loss')
        self.recon_loss_tracker   = keras.metrics.Mean(name='reconstruction_loss')
        self.kl_loss_tracker      = keras.metrics.Mean(name='kl_loss')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.recon_loss_tracker, self.kl_loss_tracker]

    def train_step(self, data):
        with tf.GradientTape() as tape:
            # Forward pass through encoder
            z_mean, z_log_var, z = self.encoder(data)
            
            # Forward pass through decoder
            reconstruction = self.decoder(z)
            
            # ── RECONSTRUCTION LOSS ──
            # Binary cross-entropy summed over all 784 pixels, averaged over batch
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(
                    binary_crossentropy(data, reconstruction),
                    axis=-1
                )
            )
            
            # ── KL DIVERGENCE LOSS ──
            # KL( N(μ, σ²) || N(0,1) ) = -0.5 * sum(1 + log(σ²) - μ² - σ²)
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )
            
            # ── TOTAL LOSS ──
            total_loss = recon_loss + kl_loss
        
        # Backpropagation
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # Update trackers
        self.total_loss_tracker.update_state(total_loss)
        self.recon_loss_tracker.update_state(recon_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            'total_loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.recon_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result()
        }

# Instantiate and compile the VAE
vae = VAE(vae_encoder, vae_decoder, name='VAE')
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3))

print("VAE model created ✅")

---
## 🏋️ Step 8: Train the VAE

In [ ]:
# Train the VAE
# Note: VAE only needs x_train_flat as input (it handles reconstruction internally)
vae_history = vae.fit(
    x_train_flat,
    epochs=20,
    batch_size=256,
    shuffle=True,
    verbose=1
)

In [ ]:
# Plot VAE loss components over training
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(vae_history.history['total_loss'], color='purple', linewidth=2)
axes[0].set_title('Total Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].grid(alpha=0.3)

axes[1].plot(vae_history.history['reconstruction_loss'], color='steelblue', linewidth=2)
axes[1].set_title('Reconstruction Loss', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].grid(alpha=0.3)

axes[2].plot(vae_history.history['kl_loss'], color='coral', linewidth=2)
axes[2].set_title('KL Divergence Loss', fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].grid(alpha=0.3)

fig.suptitle('VAE — Training Loss Components', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🖼️ Step 9: Visualize VAE Reconstruction

In [ ]:
# Get VAE reconstructions for the test set
# Step 1: Encode test images to get z_mean, z_log_var, z
z_mean_test, z_log_var_test, z_test = vae_encoder.predict(x_test_flat, verbose=0)

# Step 2: Decode z to get reconstructions
vae_reconstructed = vae_decoder.predict(z_mean_test, verbose=0)
# Note: we use z_mean (not sampled z) for deterministic, cleaner reconstructions

plot_reconstruction(x_test_flat, vae_reconstructed, 'VAE')

---
## 🗺️ Step 10: Visualize the VAE Latent Space

Compare this to the AE latent space from Step 5.

**What to look for:**  
- VAE latent space should be **smoother and more continuous** than AE
- Clusters should **overlap more** (less hard boundaries)
- The space should be **centered around (0, 0)** — the KL loss enforces this!
- Interpolating between two points in the VAE latent space gives meaningful intermediate images

In [ ]:
# Plot VAE latent space
plt.figure(figsize=(8, 6))
scatter = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1],
                      c=y_test, cmap='tab10', alpha=0.6, s=5)
plt.colorbar(scatter, label='Digit Label')
plt.title('VAE — 2D Latent Space (z_mean)', fontsize=13, fontweight='bold')
plt.xlabel('Latent Dimension 1 (μ₁)')
plt.ylabel('Latent Dimension 2 (μ₂)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 VAE latent space is denser and more centered than AE.")
print("   This means we can sample ANY point and get a valid-looking digit!")

---
## ✨ Step 11: Generate New Digits with VAE

This is the **key advantage of VAE over plain AE**: we can generate brand-new digits by sampling random points from the latent space.

We sample z ~ N(0, 1) and pass it through the decoder.  
Because the KL loss trained the latent space to follow N(0, 1), these samples produce valid-looking digits!

In [ ]:
# ── GENERATE NEW IMAGES BY SAMPLING FROM LATENT SPACE ──

# Sample 20 random latent vectors from standard normal
n_generate = 20
random_latent = np.random.normal(size=(n_generate, VAE_LATENT_DIM))

# Decode the random latent vectors
generated_images = vae_decoder.predict(random_latent, verbose=0)

# Display generated images
fig, axes = plt.subplots(2, 10, figsize=(16, 3.5))
fig.suptitle('VAE — Newly Generated Images (sampled from N(0,1))', 
             fontsize=12, fontweight='bold')

for i in range(20):
    row, col = i // 10, i % 10
    axes[row, col].imshow(generated_images[i].reshape(28, 28), cmap='gray')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()
print("🎉 These are brand-new digits — never seen in training data!")

---
## 🔀 Step 12: Latent Space Interpolation

One of the coolest features of VAE: smoothly **interpolating between two digits** in the latent space.

Pick two real images → encode to z₁ and z₂ → linearly interpolate → decode each step.

**Expected result**: a smooth transition from one digit to another (e.g., 0 → 1 → 7)

In [ ]:
# ── LATENT SPACE INTERPOLATION ──

# Pick two different test images
idx_a, idx_b = 0, 5  # You can change these indices to try different pairs

# Encode both images to get their latent means
img_a = x_test_flat[[idx_a]]
img_b = x_test_flat[[idx_b]]
z_a, _, _ = vae_encoder.predict(img_a, verbose=0)
z_b, _, _ = vae_encoder.predict(img_b, verbose=0)

# Linearly interpolate between z_a and z_b with n_steps steps
n_steps = 10
alphas = np.linspace(0, 1, n_steps)  # [0.0, 0.11, 0.22, ..., 1.0]
z_interp = np.array([(1 - a) * z_a + a * z_b for a in alphas]).squeeze()

# Decode interpolated latent vectors
interpolated_images = vae_decoder.predict(z_interp, verbose=0)

# Plot the interpolation path
fig, axes = plt.subplots(1, n_steps + 2, figsize=(18, 2.5))
fig.suptitle(f'Latent Space Interpolation: Digit {y_test[idx_a]} → Digit {y_test[idx_b]}',
             fontsize=12, fontweight='bold')

# Original image A
axes[0].imshow(img_a.reshape(28, 28), cmap='gray')
axes[0].set_title(f'Original A\n({y_test[idx_a]})', fontsize=8)
axes[0].axis('off')

# Interpolated frames
for i in range(n_steps):
    axes[i + 1].imshow(interpolated_images[i].reshape(28, 28), cmap='gray')
    axes[i + 1].set_title(f'α={alphas[i]:.1f}', fontsize=7)
    axes[i + 1].axis('off')

# Original image B
axes[-1].imshow(img_b.reshape(28, 28), cmap='gray')
axes[-1].set_title(f'Original B\n({y_test[idx_b]})', fontsize=8)
axes[-1].axis('off')

plt.tight_layout()
plt.show()

---
## ⚖️ Step 13: Side-by-Side AE vs VAE Comparison

In [ ]:
# ── DIRECT AE vs VAE COMPARISON ──

n = 8  # number of images to compare
fig, axes = plt.subplots(3, n, figsize=(16, 5))

row_labels = ['Original', 'AE Reconstruction', 'VAE Reconstruction']
data_rows  = [x_test_flat, ae_reconstructed, vae_reconstructed]

for row_idx, (label, row_data) in enumerate(zip(row_labels, data_rows)):
    for col_idx in range(n):
        ax = axes[row_idx, col_idx]
        ax.imshow(row_data[col_idx].reshape(28, 28), cmap='gray')
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(label, fontsize=10, fontweight='bold', rotation=90, labelpad=40)

plt.suptitle('Comparison: Original vs AE vs VAE Reconstruction', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── COMPUTE RECONSTRUCTION MSE ──

ae_mse  = np.mean((x_test_flat - ae_reconstructed) ** 2)
vae_mse = np.mean((x_test_flat - vae_reconstructed) ** 2)

print("📊 Reconstruction Quality (Mean Squared Error — lower is better)")
print(f"   Autoencoder MSE : {ae_mse:.5f}")
print(f"   VAE MSE         : {vae_mse:.5f}")
print()
print("💡 Note: AE often has lower MSE because it only optimizes reconstruction.")
print("   VAE trades some reconstruction quality for a structured, generative latent space.")

---
## 📚 Step 14: Summary & Conclusions

### What We Built

| | Autoencoder | VAE |
|--|--|--|
| **Architecture** | Encoder → Fixed z → Decoder | Encoder → (μ, σ²) → Sample z → Decoder |
| **Loss** | Reconstruction only | Reconstruction + KL divergence |
| **Latent Space** | Irregular, possibly fragmented | Smooth, continuous, ~ N(0,1) |
| **Can Generate?** | ❌ Poorly | ✅ Yes |
| **Reconstruction** | ✅ Slightly sharper | ✅ Good (slightly blurrier) |
| **Interpretable Interpolation** | ❌ Often breaks | ✅ Smooth transitions |

### Key Takeaways
1. **Autoencoders** are great for compression, denoising, and anomaly detection
2. **VAEs** sacrifice a little reconstruction quality to gain a *structured, generative* latent space
3. The **reparameterization trick** is the mathematical hack that makes VAE training possible
4. **KL divergence** regularizes the latent space to stay close to N(0,1), enabling generation
5. VAEs are the **conceptual foundation** for many modern generative models (VQ-VAE, diffusion models share similar ideas)

### What's Next?
- 🔹 **GANs** — A different approach to generation using adversarial training
- 🔹 **β-VAE** — Adding a β weight to KL loss for better disentanglement
- 🔹 **VQ-VAE** — Discrete latent spaces used in image generation
- 🔹 **Convolutional AE/VAE** — Use Conv layers instead of Dense for better image quality
- 🔹 **Conditional VAE (CVAE)** — Control what digit the VAE generates

---

### 🧠 Answers to Interview Questions

**Q1: What is the difference between AE and VAE?**  
AE maps input to a single point in latent space. VAE maps input to a distribution (μ, σ²) and samples from it. VAE adds KL divergence to the loss to regularize this distribution toward N(0,1).

**Q2: Why is the reparameterization trick needed?**  
Random sampling is non-differentiable, so gradients can't flow through it. By reparameterizing z = μ + σ·ε (where ε ~ N(0,1) is treated as a constant), gradients flow through μ and σ cleanly.

**Q3: What does KL divergence do?**  
It penalizes the encoder for deviating from N(0,1). This keeps the latent space smooth and organized, ensuring that nearby points decode to similar outputs — essential for generation.

**Q4: How to use AE for anomaly detection?**  
Train an AE on normal data. Anomalies (unseen patterns) will have high reconstruction error. Use reconstruction error as an anomaly score — high error = likely anomaly.

**Q5: Why is VAE's latent space better for generation?**  
Because KL loss guarantees the latent space is continuous and every region maps to a valid output. In plain AE, you could sample a gap between clusters and get garbage output.